# Derivation: Computing the Parts of the Sector Factor Model
The [lecture](../../CHEME-5660-L13b-Lecture-JumpHMM-ManyStocks-Fall-2026.ipynb) gives the results that compute the parts of the sector factor model from the data. Where do the sector factor, the sector loading, and the residual scale come from?

> __Learning Objectives:__
>
> By the end of this derivation, you should be able to:
> * __Remove the market from the sector factor:__ Average the single index residuals of the stocks in a sector, and substitute the single index fit of each stock. Show that the average keeps the growth rate of the sector and removes its market part.
> * __Estimate the sector loading:__ Set up the least-squares fit of a stock's single index residual on the sector factor computed without that stock. Scale the slopes of each sector so that the average covariance of two residuals in the sector matches the data.
> * __Size the stock-specific draw:__ Show that the stock-specific residual averages zero in the data. Derive the residual scale that keeps the variance of each single index residual when the sector and stock-specific draws are independent.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13b) lists these symbols with their meanings and units.

Let's get started!
___

## The Sector Factor
The single index fit of stock $i$ gives its baseline $\alpha_{i}$, its market sensitivity $\beta_{i}$, and its single index residual $\varepsilon_{i}(t)=g_{i}(t)-\alpha_{i}-\beta_{i}\,g_{m}(t)$. The sector factor $f_{s}(t)$ is the average single index residual of the $n_{s}$ stocks in sector $s$. Substituting the single index fit of each stock into this average gives:

$$
\begin{align*}
f_{s}(t) &= \frac{1}{n_{s}}\sum_{j\in s}\varepsilon_{j}(t) && \text{average single index residual}\\
&= \frac{1}{n_{s}}\sum_{j\in s}\left(g_{j}(t)-\alpha_{j}-\beta_{j}\,g_{m}(t)\right) && \text{single index fit of each stock } j\\
&= \frac{1}{n_{s}}\sum_{j\in s}g_{j}(t)-\frac{1}{n_{s}}\sum_{j\in s}\alpha_{j}-\left(\frac{1}{n_{s}}\sum_{j\in s}\beta_{j}\right)g_{m}(t) && \text{split the sum, and factor out } g_{m}(t)\\
&= \underbrace{\bar{g}_{s}(t)}_{\text{sector average}}-\underbrace{\bar{\alpha}_{s}}_{\text{baseline}}-\underbrace{\bar{\beta}_{s}\,g_{m}(t)}_{\text{market}} && \text{name the three averages}
\end{align*}
$$

Each bar is an average over the $n_{s}$ stocks in the sector: $\bar{g}_{s}(t)$ of their growth rates on day $t$, $\bar{\alpha}_{s}$ of their baselines, and $\bar{\beta}_{s}$ of their market sensitivities. If the sector factor kept the market part, a market move would enter the growth rate twice, once through $\beta_{i}\,g_{m}(t)$ and once through $\delta_{i}\,f_{s}(t)$.

## The Sector Loading
The sector loading $\delta_{i}$ measures how strongly the single index residual of stock $i$ moves with its sector factor, as $\beta_{i}$ measures how strongly the stock moves with the market. We estimate $\delta_{i}$ by least squares, as we did for $\alpha_{i}$ and $\beta_{i}$. The regressor is the sector factor computed without stock $i$, the average single index residual of the other $n_{s}-1$ stocks:

$$
f_{s,-i}(t) = \frac{1}{n_{s}-1}\sum_{j\in s,\,j\neq i}\varepsilon_{j}(t).
$$

We leave stock $i$ out of this average, so that its own single index residual does not appear on both sides of the fit. Both $\varepsilon_{i}(t)$ and $f_{s,-i}(t)$ average zero in the data, so the fit needs no intercept. Stacking the fit for stock $i$ over the $T$ trading days of the data gives:

$$
\underbrace{\begin{bmatrix} \varepsilon_{i}(1)\\ \vdots\\ \varepsilon_{i}(T)\end{bmatrix}}_{\substack{\mathbf{y}\\ \text{single index residual of stock } i}}
=
\underbrace{\begin{bmatrix} f_{s,-i}(1)\\ \vdots\\ f_{s,-i}(T)\end{bmatrix}}_{\substack{\hat{\mathbf{X}}\\ \text{sector factor without stock } i}}
\,\hat{\delta}_{i}
+
\underbrace{\mathbf{e}}_{\text{fit errors}}
$$

This is an overdetermined system, with $T$ equations and one unknown. No single value of $\delta$ makes every error zero. Instead, the least-squares estimate minimizes the sum of squared errors, which has an analytical solution:

$$
\boxed{
\hat{\delta}_{i} = \arg\min_{\delta}\;\lVert\mathbf{y}-\hat{\mathbf{X}}\,\delta\rVert_{2}^{2}
\quad\Longrightarrow\quad
\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)\hat{\delta}_{i} = \hat{\mathbf{X}}^{\top}\mathbf{y}
\quad\Longrightarrow\quad
\hat{\delta}_{i} = \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}
}
$$

Because both series average zero, the solution is the sample covariance over the sample variance, like $\hat{\beta}_{i}$:

$$
\hat{\delta}_{i}
= \frac{\sum_{t=1}^{T}f_{s,-i}(t)\,\varepsilon_{i}(t)}{\sum_{t=1}^{T}f_{s,-i}(t)^{2}}
= \frac{\widehat{\operatorname{Cov}}\left(\varepsilon_{i},f_{s,-i}\right)}{\widehat{\operatorname{Var}}\left(f_{s,-i}\right)}.
$$

We keep only the slope $\hat{\delta}_{i}$ from this fit and discard the fit errors. We then multiply the slopes of each sector by one number $c_{s}$, so the sector loading is $\delta_{i} = c_{s}\,\hat{\delta}_{i}$. We choose $c_{s}$ so that the off-diagonal entries of the sector block match the residual covariances in the data on average.

__Choosing the scale:__ The sector factor model gives two different stocks $i$ and $j$ of sector $s$ the residual covariance $\delta_{i}\delta_{j}\operatorname{Var}(f_{s})$, the off-diagonal entry of the sector block. We choose $c_{s}$ so that the average of these entries over the $n_{s}(n_{s}-1)$ ordered pairs equals the average residual covariance of the same pairs in the data:

$$
\begin{align*}
\frac{1}{n_{s}(n_{s}-1)}\sum_{i\neq j}c_{s}^{2}\,\hat{\delta}_{i}\hat{\delta}_{j}\operatorname{Var}(f_{s}) &= \frac{1}{n_{s}(n_{s}-1)}\sum_{i\neq j}\operatorname{Cov}(\varepsilon_{i},\varepsilon_{j}) && \text{match the averages, with } \delta_{i}=c_{s}\,\hat{\delta}_{i}\\
c_{s} &= \sqrt{\frac{\sum_{i\neq j}\operatorname{Cov}(\varepsilon_{i},\varepsilon_{j})}{\operatorname{Var}(f_{s})\sum_{i\neq j}\hat{\delta}_{i}\hat{\delta}_{j}}} && \text{solve for } c_{s}
\end{align*}
$$

Both sums run over the pairs of stocks in sector $s$. The sum of the residual covariances follows from the variance of the sector factor, because $n_{s}\,f_{s}(t)$ is the sum of the single index residuals of the sector:

$$
\begin{align*}
n_{s}^{2}\operatorname{Var}(f_{s}) &= \operatorname{Var}\left(\sum_{i\in s}\varepsilon_{i}\right) && n_{s}\,f_{s}=\textstyle\sum_{i\in s}\varepsilon_{i}\\
&= \sum_{i\in s}\operatorname{Var}(\varepsilon_{i})+\sum_{i\neq j}\operatorname{Cov}(\varepsilon_{i},\varepsilon_{j}) && \text{variance of a sum}\\
\sum_{i\neq j}\operatorname{Cov}(\varepsilon_{i},\varepsilon_{j}) &= n_{s}^{2}\operatorname{Var}(f_{s})-\sum_{i\in s}\operatorname{Var}(\varepsilon_{i}) && \text{solve for the sum of the covariances}
\end{align*}
$$

In the same way, the sum of the products of the slopes is $\sum_{i\neq j}\hat{\delta}_{i}\hat{\delta}_{j}=\left(\sum_{i\in s}\hat{\delta}_{i}\right)^{2}-\sum_{i\in s}\hat{\delta}_{i}^{2}$. Both sums must be positive, so the stocks of each sector must still move together after we remove the market.

Removing the sector part from the single index residual of stock $i$ leaves the stock-specific residual $u_{i}(t) = \varepsilon_{i}(t)-\delta_{i}\,f_{s}(t)$.

## The Stock-Specific Draw
We fit a jump hidden Markov model to the stock-specific residual $u_{i}(t)$ of each stock, the same way we fit and tuned the model of `SPY`. A path simulated from this stock-specific model is the stock-specific draw $u_{i}^{\mathrm{draw}}(t)$. We multiply the draw by the residual scale $\psi_{i}$ and add the stock-specific term $\psi_{i}\,u_{i}^{\mathrm{draw}}(t)$ to the growth rate of stock $i$.

__The stock-specific residual averages zero:__ The single index fit chooses $\alpha_{i}$ to minimize the sum of squared errors. Setting the derivative of this sum with respect to $\alpha_{i}$ to zero gives $-2\sum_{t=1}^{T}\varepsilon_{i}(t)=0$. Averaging $u_{i}(t)$ over the $T$ days of the data then shows that the stock-specific residual averages zero:

$$
\begin{align*}
\frac{1}{T}\sum_{t=1}^{T}u_{i}(t) &= \frac{1}{T}\sum_{t=1}^{T}\varepsilon_{i}(t)-\delta_{i}\,\frac{1}{T}\sum_{t=1}^{T}f_{s}(t) && u_{i}(t)=\varepsilon_{i}(t)-\delta_{i}\,f_{s}(t)\\
&= \frac{1}{T}\sum_{t=1}^{T}\varepsilon_{i}(t)-\frac{\delta_{i}}{n_{s}}\sum_{j\in s}\left(\frac{1}{T}\sum_{t=1}^{T}\varepsilon_{j}(t)\right) && f_{s}(t)\text{ is the average single index residual}\\
&= 0 && \text{every single index residual sums to zero}
\end{align*}
$$

The stock-specific model is fit to this zero-mean series. A long path drawn from the model also averages close to zero, so the stock-specific term $\psi_{i}\,u_{i}^{\mathrm{draw}}(t)$ does not change the baseline $\alpha_{i}$ of stock $i$.

__Sizing the stock-specific draw:__ In the data, the single index residual of stock $i$ splits into its sector part and its stock-specific residual, $\varepsilon_{i}(t)=\delta_{i}\,f_{s}(t)+u_{i}(t)$. The variance of this sum is:

$$
\operatorname{Var}(\varepsilon_{i}) = \delta_{i}^{2}\operatorname{Var}(f_{s}) + \operatorname{Var}(u_{i}) + 2\,\delta_{i}\operatorname{Cov}(f_{s},u_{i}).
$$

In the simulation, the sector draw $f_{s}^{\mathrm{draw}}(t)$ and the stock-specific draw $u_{i}^{\mathrm{draw}}(t)$ come from separate models, drawn independently. The simulated single index residual of stock $i$ is $\delta_{i}\,f_{s}^{\mathrm{draw}}(t)+\psi_{i}\,u_{i}^{\mathrm{draw}}(t)$. The variance of this simulated residual is:

$$
\begin{align*}
\operatorname{Var}\left(\delta_{i}\,f_{s}^{\mathrm{draw}}+\psi_{i}\,u_{i}^{\mathrm{draw}}\right) &= \delta_{i}^{2}\operatorname{Var}(f_{s}^{\mathrm{draw}}) + \psi_{i}^{2}\operatorname{Var}(u_{i}^{\mathrm{draw}}) && \text{independent draws: the covariance is zero}\\
&\approx \delta_{i}^{2}\operatorname{Var}(f_{s}) + \psi_{i}^{2}\operatorname{Var}(u_{i}) && \text{each fitted model matches the variance of its data}
\end{align*}
$$

We choose $\psi_{i}$ so that this variance equals the variance of the single index residual in the data, the diagonal entry of the residual matrix $\mathbf{\Sigma}_{\varepsilon}$ in the lecture:

$$
\begin{align*}
\delta_{i}^{2}\operatorname{Var}(f_{s}) + \psi_{i}^{2}\operatorname{Var}(u_{i}) &= \operatorname{Var}(\varepsilon_{i}) && \text{match the data}\\
\psi_{i}^{2} &= \boxed{\frac{\operatorname{Var}(\varepsilon_{i})-\delta_{i}^{2}\operatorname{Var}(f_{s})}{\operatorname{Var}(u_{i})}} && \text{solve for } \psi_{i}^{2}\\
&= \frac{\delta_{i}^{2}\operatorname{Var}(f_{s}) + \operatorname{Var}(u_{i}) + 2\,\delta_{i}\operatorname{Cov}(f_{s},u_{i})-\delta_{i}^{2}\operatorname{Var}(f_{s})}{\operatorname{Var}(u_{i})} && \text{variance of } \varepsilon_{i} \text{ in the data}\\
&= 1 + \frac{2\,\delta_{i}\operatorname{Cov}(f_{s},u_{i})}{\operatorname{Var}(u_{i})} && \text{cancel } \delta_{i}^{2}\operatorname{Var}(f_{s})
\end{align*}
$$

We take the positive root for $\psi_{i}$. The last line shows that the scale is one when $u_{i}$ and $f_{s}$ do not move together in the data. When they move together, the scale is above one, and the larger stock-specific draw makes up for the covariance term that the independent draws drop.

To see when they move together, we expand the covariance with $u_{i}=\varepsilon_{i}-\delta_{i}\,f_{s}$:

$$
\begin{align*}
\operatorname{Cov}(f_{s},u_{i}) &= \operatorname{Cov}(f_{s},\varepsilon_{i}) - \delta_{i}\operatorname{Var}(f_{s}) && u_{i}=\varepsilon_{i}-\delta_{i}\,f_{s}\\
&= \operatorname{Var}(f_{s})\left(\frac{\operatorname{Cov}(f_{s},\varepsilon_{i})}{\operatorname{Var}(f_{s})}-\delta_{i}\right) && \text{factor out } \operatorname{Var}(f_{s})
\end{align*}
$$

The ratio inside the parentheses is the least-squares slope of $\varepsilon_{i}$ on the full sector factor $f_{s}$, stock $i$ included. The sector loading $\delta_{i}$ comes from the fit without stock $i$, scaled by $c_{s}$, so the loading differs from this slope. As a result, the covariance is not zero, and the scale $\psi_{i}$ differs from one.

## Summary
We derived the sector factor, the sector loading, and the residual scale of the sector factor model from the single index fit of each stock.

> __Key Takeaways:__
>
> * __The sector factor has no market part:__ Averaging the single index residuals of a sector subtracts the average baseline and the average market part from the growth rate of the sector. A market move enters each growth rate once, through the market term.
> * __The sector loading is a scaled least-squares slope:__ We fit each stock's single index residual on the sector factor computed without that stock, and kept the slope. One scale per sector makes the average covariance of two residuals in the sector match the data.
> * __The residual scale keeps the variance:__ The stock-specific residual averages zero in the data, and its model is fit to that series. The residual scale makes up for the covariance between the sector factor and the stock-specific residual, which the independent draws drop.

Return to the [lecture](../../CHEME-5660-L13b-Lecture-JumpHMM-ManyStocks-Fall-2026.ipynb), or explore the [sector factor example](../../CHEME-5660-L13b-Example-JumpHMM-SectorFactor-Fall-2026.ipynb), which computes these parts for a universe of stocks.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___